<a href="https://colab.research.google.com/github/labdall4/CPS_Project/blob/main/RPS_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Evaluation File**

Cell 1: Header and Shared Spec Constants

In [3]:
# RPS 1 evaluation notebook
# owner evaluation follows spec ipynb v1 1

import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# copied from spec ipynb section 1 edit them in the spec not here
SHAPES = ['Rock', 'Paper', 'Scissors']
PLAYERS = ['self', 'opponent']
NUM_HANDS = 4
SEED = 42
PAYOFF = {'win': 1, 'tie': 0, 'loss': -1}
BEATS = {'Rock': 'Scissors', 'Scissors': 'Paper', 'Paper': 'Rock'}

Cell 2: Import Contracts from SPEC, RPS_detection, and RPS_strategy

In [4]:
# pull in decide and detect hands from the other notebooks
try:
    %run -i RPS_strategy.ipynb
except Exception as e:
    print('Could not load RPS_strategy.ipynb:', e)

try:
    %run -i RPS_detection.ipynb
except Exception as e:
    print('RPS_detection.ipynb not ready yet:', e)

# fallback validator in case spec ipynb wasnt run first
if 'check_hands' not in globals():
    def check_hands(hands):
        if len(hands) != NUM_HANDS:
            raise ValueError('expected 4 hands, got ' + str(len(hands)))
        if sorted(h['position'] for h in hands) != [0, 1, 2, 3]:
            raise ValueError('positions must be 0 to 3, each used once')
        for h in hands:
            if h['player'] not in PLAYERS:
                raise ValueError('bad player: ' + str(h['player']))
            if h['shape'] not in SHAPES:
                raise ValueError('bad shape: ' + str(h['shape']))
        counts = {p: sum(1 for h in hands if h['player'] == p) for p in PLAYERS}
        if counts['self'] != 2 or counts['opponent'] != 2:
            raise ValueError('need 2 hands per player, got ' + str(counts))
        return True

Could not load RPS_strategy.ipynb: File `'RPS_strategy.ipynb.py'` not found.
RPS_detection.ipynb not ready yet: File `'RPS_detection.ipynb.py'` not found.


Cell 3: Detection Evaluation Against data/expected_labels.json

In [5]:
def evaluate_detection(image_dir='data/demo', labels_path='data/expected_labels.json'):
    labels_file = Path(labels_path)
    if not labels_file.exists():
        raise FileNotFoundError(f'Missing labels file: {labels_file}')

    with open(labels_file, 'r') as f:
        expected_map = json.load(f)

    img_dir = Path(image_dir)
    image_files = sorted([
        p for p in img_dir.iterdir()
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and p.name in expected_map
    ])

    if not image_files:
        print(f'No labeled images found in {image_dir}.')
        return None

    shape_to_idx = {s: i for i, s in enumerate(SHAPES)}
    confusion = np.zeros((3, 3), dtype=int)  # rows are ground truth cols are predicted

    total_images = len(image_files)
    valid_images = 0
    exact_images = 0
    total_hands = 0
    player_correct = 0
    shape_correct = 0
    failures = []

    for img_path in image_files:
        expected_hands = sorted(expected_map[img_path.name], key=lambda h: h['position'])
        try:
            pred_hands = detect_hands(str(img_path))
            check_hands(pred_hands)
            valid_images += 1
        except Exception as exc:
            failures.append((img_path.name, str(exc)))
            total_hands += NUM_HANDS
            continue

        all_four_correct = True
        for pred_h, exp_h in zip(pred_hands, expected_hands):
            total_hands += 1
            p_ok = (pred_h['player'] == exp_h['player'])
            s_ok = (pred_h['shape'] == exp_h['shape'])

            if p_ok:
                player_correct += 1
            if s_ok:
                shape_correct += 1
            if not (p_ok and s_ok):
                all_four_correct = False

            confusion[shape_to_idx[exp_h['shape']], shape_to_idx[pred_h['shape']]] += 1

        if all_four_correct:
            exact_images += 1

        # run the detected hands through decide to verify the full pipeline
        if 'decide' in globals():
            self_h = [h for h in pred_hands if h['player'] == 'self']
            opp_h = [h for h in pred_hands if h['player'] == 'opponent']
            decision = decide(self_h, opp_h, rng=np.random.default_rng(SEED))
            print(f"{img_path.name} -> keep {decision['keep_shape']} (pos {decision['keep_position']}, EV={decision['expected_payoff']:+.2f})")

    return {
        'split': image_dir,
        'total_images': total_images,
        'valid_contract_images': valid_images,
        'exact_match_images': exact_images,
        'player_accuracy': player_correct / max(total_hands, 1),
        'shape_accuracy': shape_correct / max(total_hands, 1),
        'failures': failures,
        'confusion_matrix': confusion,
    }

# check demo images first
if Path('data/demo').exists() and Path('data/expected_labels.json').exists() and 'detect_hands' in globals():
    demo_results = evaluate_detection('data/demo', 'data/expected_labels.json')
    if demo_results:
        print(f"\nSplit: {demo_results['split']}")
        print(f"Valid 4-hand outputs: {demo_results['valid_contract_images']}/{demo_results['total_images']}")
        print(f"Exact image matches:  {demo_results['exact_match_images']}/{demo_results['total_images']}")
        print(f"Player accuracy:      {demo_results['player_accuracy']:.1%}")
        print(f"Shape accuracy:       {demo_results['shape_accuracy']:.1%}")
        if demo_results['failures']:
            print('Failures:', demo_results['failures'])
else:
    print('Skipping demo image check (waiting on data/demo, expected_labels.json, or detect_hands).')

Skipping demo image check (waiting on data/demo, expected_labels.json, or detect_hands).


Cell 4: End-to-End Integration and Payoff Simulation

In [6]:
def round_score(self_shape, opp_shape):
    if self_shape == opp_shape:
        return PAYOFF['tie']
    return PAYOFF['win'] if BEATS[self_shape] == opp_shape else PAYOFF['loss']

def evaluate_strategy_payoffs(n_rounds=10000, seed=SEED):
    rng = np.random.default_rng(seed)

    # worked example from the brief us is paper and scissors opponent is paper and rock
    self_hands = [
        {'player': 'self', 'shape': 'Paper', 'position': 0, 'confidence': 0.95},
        {'player': 'self', 'shape': 'Scissors', 'position': 1, 'confidence': 0.90},
    ]
    opp_hands = [
        {'player': 'opponent', 'shape': 'Paper', 'position': 2, 'confidence': 0.88},
        {'player': 'opponent', 'shape': 'Rock', 'position': 3, 'confidence': 0.93},
    ]

    # against an optimal opponent 2 3 paper 1 3 rock
    nash_scores = []
    for _ in range(n_rounds):
        action = decide(self_hands, opp_hands, rng=rng)
        opp_shape = 'Paper' if rng.random() < (2 / 3) else 'Rock'
        nash_scores.append(round_score(action['keep_shape'], opp_shape))

    # against a predictable opponent who always plays paper to avoid losing
    naive_scores = []
    for _ in range(n_rounds):
        action = decide(self_hands, opp_hands, rng=rng)
        naive_scores.append(round_score(action['keep_shape'], 'Paper'))

    print(f"PS vs PR over {n_rounds} rounds:")
    print(f"  vs Nash opponent:         {np.mean(nash_scores):+.4f} (expected +0.3333)")
    print(f"  vs Always-Paper opponent: {np.mean(naive_scores):+.4f} (expected +0.3333)")

    assert abs(np.mean(nash_scores) - (1 / 3)) < 0.03
    assert abs(np.mean(naive_scores) - (1 / 3)) < 0.03
    return nash_scores, naive_scores

nash_scores, naive_scores = evaluate_strategy_payoffs()

NameError: name 'decide' is not defined

Cell 5: Plots (Cumulative Average Payoff and Confusion Matrix)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# running average payoff over time
rounds = np.arange(1, len(nash_scores) + 1)
axes[0].plot(rounds, np.cumsum(nash_scores) / rounds, label='vs Nash opponent')
axes[0].plot(rounds, np.cumsum(naive_scores) / rounds, label='vs Always-Paper opponent', alpha=0.8)
axes[0].axhline(1 / 3, color='black', linestyle='--', label='Expected EV (1/3)')
axes[0].set_title('Empirical Payoff Convergence (PS vs PR)')
axes[0].set_xlabel('Round')
axes[0].set_ylabel('Average Payoff')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# shape confusion matrix
cm = demo_results['confusion_matrix'] if ('demo_results' in globals() and demo_results) else np.zeros((3, 3), dtype=int)
axes[1].imshow(cm, cmap='Blues')
axes[1].set_xticks(range(3))
axes[1].set_yticks(range(3))
axes[1].set_xticklabels(SHAPES)
axes[1].set_yticklabels(SHAPES)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Expected')
axes[1].set_title('Shape Confusion Matrix')
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, str(cm[i, j]), ha='center', va='center', color='black')

plt.tight_layout()
plt.show()

Cell 6: Separate Test Set Evaluation & Environment Freeze (SPEC Sections 4 & 5)

In [ ]:
# only run once detection is finalized never tune on data test
if Path('data/test').exists() and Path('data/expected_labels.json').exists() and 'detect_hands' in globals():
    test_results = evaluate_detection('data/test', 'data/expected_labels.json')
    if test_results:
        print('Test results:', {k: v for k, v in test_results.items() if k != 'confusion_matrix'})

# record installed versions per spec section 5
!pip freeze